# 02 - Pipeline Monitoring

Operational dashboard for the BigQuery warehouse. Reads the live tables and renders the signals an on-call data engineer checks after each daily build: row-count reconciliation across layers, freshness against the SLO, data-quality failures by reason, future-dated drift, and the quarantine rate trend.

All identifiers are set in the parameter cell so the notebook can be pointed at any project or environment.

**Sections**
1. Row-count audit per layer
2. Freshness check
3. DQ failure breakdown
4. Future-dated monitoring
5. Quarantine rate over time
6. SLO dashboard summary

**Auth.** Application Default Credentials (`gcloud auth application-default login` locally; attached service account on Cloud Run).

In [1]:
# --- Parameters: edit these for your environment ---
PROJECT_ID = 'hypefast-data-prod'
RAW_DATASET = 'raw'
STAGING_DATASET = 'staging'
MARTS_DATASET = 'marts'
QUARANTINE_DATASET = 'quarantine'

# SLO thresholds
FRESHNESS_SLO_HOURS = 24        # mart staleness budget
FRESHNESS_WARN_HOURS = 18       # early-warning band
QUARANTINE_RATE_THRESHOLD = 0.05  # 5% rejected rows is the alerting line

print('Project:', PROJECT_ID)

Project: hypefast-data-prod


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

# Fully-qualified table references built from the parameters.
T_RAW_TXN = f'`{PROJECT_ID}.{RAW_DATASET}.raw_transactions`'
T_RAW_CUST = f'`{PROJECT_ID}.{RAW_DATASET}.raw_customers`'
T_RAW_BRANCH = f'`{PROJECT_ID}.{RAW_DATASET}.raw_branches`'
T_STG_TXN = f'`{PROJECT_ID}.{STAGING_DATASET}.stg_transactions`'
T_FACT = f'`{PROJECT_ID}.{MARTS_DATASET}.fact_transactions`'
T_QUARANTINE = f'`{PROJECT_ID}.{QUARANTINE_DATASET}.transactions_rejected`'

def run_query(sql):
    """Execute SQL and return a pandas DataFrame."""
    return client.query(sql).to_dataframe()

print('BigQuery client ready.')

/Users/hkn000112/Documents/Firza/Recruitment/Astra Financial - Connectix/retail-financing-platform/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.13) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


BigQuery client ready.


## Section 1: Row-count audit per layer

Reconciles volume across raw, staging, fact, and quarantine. Expected on the Phase 1 dataset: raw 12,300 (with 300 duplicate ids) deduplicates to 12,000 in staging and fact. Quarantine holds the rows failing a hard DQ rule; the fact table carries the same rows flagged `is_quarantined = TRUE` so totals stay complete.

In [3]:
row_audit_sql = f"""
SELECT 'raw.raw_transactions'         AS layer, COUNT(*) AS row_count FROM {T_RAW_TXN}
UNION ALL SELECT 'staging.stg_transactions', COUNT(*) FROM {T_STG_TXN}
UNION ALL SELECT 'marts.fact_transactions',  COUNT(*) FROM {T_FACT}
UNION ALL SELECT 'marts.fact (quarantined)', COUNTIF(is_quarantined) FROM {T_FACT}
UNION ALL SELECT 'quarantine.transactions_rejected', COUNT(*) FROM {T_QUARANTINE}
ORDER BY row_count DESC
"""
row_audit = run_query(row_audit_sql)
display(row_audit)

fig = px.bar(row_audit, x='layer', y='row_count', text='row_count',
             title='Row count by pipeline layer', color='layer')
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, xaxis_tickangle=-30, height=420)
fig.show()

/Users/hkn000112/Documents/Firza/Recruitment/Astra Financial - Connectix/retail-financing-platform/.venv/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,layer,row_count
0,raw.raw_transactions,12300
1,marts.fact_transactions,12000
2,staging.stg_transactions,12000
3,marts.fact (quarantined),427
4,quarantine.transactions_rejected,427


## Section 2: Freshness check

Mart staleness measured as the gap between `MAX(updated_at)` in `fact_transactions` and now. Status follows the SLO bands set in the parameter cell (OK / WARN / BREACH).

In [4]:
freshness_sql = f"""
SELECT
    MAX(updated_at)        AS last_updated_at,
    MAX(transaction_ts)    AS last_transaction_ts,
    TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), MAX(updated_at), HOUR) AS hours_since_updated,
    CASE
        WHEN TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), MAX(updated_at), HOUR) > {FRESHNESS_SLO_HOURS}  THEN 'BREACH'
        WHEN TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), MAX(updated_at), HOUR) > {FRESHNESS_WARN_HOURS} THEN 'WARN'
        ELSE 'OK'
    END AS status
FROM {T_FACT}
"""
freshness = run_query(freshness_sql)
display(freshness)

hours_since = float(freshness['hours_since_updated'].iloc[0])
fresh_status = freshness['status'].iloc[0]
_color = {'OK': 'green', 'WARN': 'orange', 'BREACH': 'red'}[fresh_status]

gauge = go.Figure(go.Indicator(
    mode='gauge+number',
    value=hours_since,
    title={'text': f'Hours since last update (status: {fresh_status})'},
    gauge={
        'axis': {'range': [0, max(FRESHNESS_SLO_HOURS * 2, hours_since * 1.2)]},
        'bar': {'color': _color},
        'steps': [
            {'range': [0, FRESHNESS_WARN_HOURS], 'color': '#d9f0d3'},
            {'range': [FRESHNESS_WARN_HOURS, FRESHNESS_SLO_HOURS], 'color': '#fee08b'},
            {'range': [FRESHNESS_SLO_HOURS, max(FRESHNESS_SLO_HOURS * 2, hours_since * 1.2)], 'color': '#f4a582'},
        ],
        'threshold': {'line': {'color': 'red', 'width': 3}, 'value': FRESHNESS_SLO_HOURS},
    },
))
gauge.update_layout(height=350)
gauge.show()

/Users/hkn000112/Documents/Firza/Recruitment/Astra Financial - Connectix/retail-financing-platform/.venv/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,last_updated_at,last_transaction_ts,hours_since_updated,status
0,2026-06-07 16:32:00+00:00,2026-06-05 23:27:00+00:00,-75,OK


## Section 3: DQ failure breakdown

Quarantined rows aggregated by reason. `dq_reason` is pipe-delimited (a row can fail several rules), so it is split and counted per reason. Expected on the Phase 1 dataset: null_amount 179, negative_amount 84, broken_fk_customer 80, broken_fk_branch 80, future_date 64, status_amount_conflict 27.

In [5]:
dq_sql = f"""
WITH exploded AS (
    SELECT reason
    FROM {T_QUARANTINE}, UNNEST(SPLIT(dq_reason, '|')) AS reason
    WHERE reason != ''
)
SELECT
    reason,
    COUNT(*) AS occurrences,
    SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()) AS pct_of_total
FROM exploded
GROUP BY reason
ORDER BY occurrences DESC
"""
dq = run_query(dq_sql)
display(dq)

fig = px.bar(dq, x='reason', y='occurrences', text='occurrences',
             title='Quarantined rows by DQ reason', color='reason')
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, xaxis_tickangle=-30, height=420)
fig.show()

/Users/hkn000112/Documents/Firza/Recruitment/Astra Financial - Connectix/retail-financing-platform/.venv/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,reason,occurrences,pct_of_total
0,null_amount,172,0.377193
1,negative_amount,81,0.177632
2,broken_fk_branch,80,0.175439
3,broken_fk_customer,78,0.171053
4,status_amount_conflict,26,0.057018
5,future_date,19,0.041667


## Section 4: Future-dated monitoring

Counts fact rows with `transaction_ts > NOW()` and shows where they land by date. These are kept (flagged, not dropped); a sudden jump signals an upstream clock or load-date problem. The count is date-relative and drifts as the run date advances.

In [6]:
future_sql = f"""
SELECT
    DATE(transaction_ts) AS transaction_date,
    COUNT(*)             AS future_rows
FROM {T_FACT}
WHERE transaction_ts > CURRENT_TIMESTAMP()
GROUP BY transaction_date
ORDER BY transaction_date
"""
future = run_query(future_sql)
total_future = int(future['future_rows'].sum()) if not future.empty else 0
print(f'fact rows with transaction_ts > NOW(): {total_future}')
display(future)

if not future.empty:
    fig = px.line(future, x='transaction_date', y='future_rows', markers=True,
                  title='Future-dated fact rows by transaction date')
    fig.update_layout(height=380)
    fig.show()
else:
    print('No future-dated rows.')

fact rows with transaction_ts > NOW(): 29


/Users/hkn000112/Documents/Firza/Recruitment/Astra Financial - Connectix/retail-financing-platform/.venv/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,transaction_date,future_rows
0,2026-06-04,10
1,2026-06-05,19


## Section 5: Quarantine rate over time

Rejected rows as a share of the total processed, per build day. Build day is taken from the dbt load timestamps (`rejected_at` on quarantine, `loaded_at` on fact), which align per run. A rising rate means source quality is degrading.

In [7]:
quarantine_rate_sql = f"""
WITH rejected AS (
    SELECT DATE(rejected_at) AS batch_date, COUNT(*) AS rejected_rows
    FROM {T_QUARANTINE}
    GROUP BY batch_date
),
loaded AS (
    SELECT DATE(loaded_at) AS batch_date, COUNT(*) AS total_rows
    FROM {T_FACT}
    GROUP BY batch_date
)
SELECT
    COALESCE(l.batch_date, r.batch_date) AS batch_date,
    COALESCE(l.total_rows, 0)            AS total_rows,
    COALESCE(r.rejected_rows, 0)         AS rejected_rows,
    SAFE_DIVIDE(COALESCE(r.rejected_rows, 0), NULLIF(l.total_rows, 0)) AS quarantine_rate
FROM loaded l
FULL OUTER JOIN rejected r USING (batch_date)
ORDER BY batch_date
"""
q_rate = run_query(quarantine_rate_sql)
display(q_rate)

if not q_rate.empty:
    fig = px.line(q_rate, x='batch_date', y='quarantine_rate', markers=True,
                  title='Quarantine rate per build day')
    fig.add_hline(y=QUARANTINE_RATE_THRESHOLD, line_dash='dash', line_color='red',
                  annotation_text=f'threshold {QUARANTINE_RATE_THRESHOLD:.0%}')
    fig.update_layout(height=380, yaxis_tickformat='.1%')
    fig.show()
else:
    print('No build history yet.')

/Users/hkn000112/Documents/Firza/Recruitment/Astra Financial - Connectix/retail-financing-platform/.venv/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,batch_date,total_rows,rejected_rows,quarantine_rate
0,2026-06-04,12000,427,0.035583


## Section 6: SLO dashboard summary

One-glance status: freshness against the SLO and the latest quarantine rate against its threshold. Green = within budget, red = breached.

In [8]:
latest_rate = float(q_rate['quarantine_rate'].dropna().iloc[-1]) if not q_rate['quarantine_rate'].dropna().empty else 0.0
rate_breached = latest_rate > QUARANTINE_RATE_THRESHOLD
freshness_breached = fresh_status == 'BREACH'

summary = pd.DataFrame([
    {'slo': 'Freshness (hours)', 'value': round(hours_since, 1),
     'threshold': FRESHNESS_SLO_HOURS, 'status': fresh_status},
    {'slo': 'Quarantine rate', 'value': round(latest_rate, 4),
     'threshold': QUARANTINE_RATE_THRESHOLD, 'status': 'BREACH' if rate_breached else 'OK'},
])

def _style(row):
    color = '#f4a582' if row['status'] == 'BREACH' else ('#fee08b' if row['status'] == 'WARN' else '#d9f0d3')
    return [f'background-color: {color}'] * len(row)

display(summary.style.apply(_style, axis=1))

overall = 'BREACH' if (freshness_breached or rate_breached) else 'OK'
print(f'\nOverall pipeline SLO status: {overall}')
print(f"  freshness: {fresh_status} ({hours_since:.1f}h vs {FRESHNESS_SLO_HOURS}h budget)")
print(f"  quarantine rate: {latest_rate:.2%} vs {QUARANTINE_RATE_THRESHOLD:.0%} threshold")

,slo,value,threshold,status
0,Freshness (hours),-75.000000,24.000000,OK
1,Quarantine rate,0.035600,0.050000,OK



Overall pipeline SLO status: OK
  freshness: OK (-75.0h vs 24h budget)
  quarantine rate: 3.56% vs 5% threshold
